# Autoregressive Conditional Heteroskedasticity (ARCH)
- periods of high volatility are followed by periods of even higher volatility
- periods of low volatility are followed by periods of even lower volatility

In practice this means that volatility tends to cluster.

## GARCH
- error varince is thought to be autocorrelated over time
- assume the variance of the error term follows a process based on an autoregressive moving average
- supposed to capture better long-term volatility patterns

## ARCH(q) == GARCH(0, q)

In [10]:
import numpy as np

returns = np.array([0.01, -0.02, 0.015, -0.005, 0.007])
T = len(returns)

for i in range(100001):
    # Randomizeparameters near some base values
    omega = 1e-6 * (1 + 0.1 * np.random.randn())
    alpha = 0.1 + 0.02 * np.random.randn()
    beta = 0.85 + 0.02 * np.random.randn()

    sigma2 = np.zeros(T)
    eps = returns

    sigma2[0] = np.var(returns)

    g_omega = 0.0
    g_alpha = 0.0
    g_beta = 0.0

    loglikelihood = 0.0
    score_omega = 0.0
    score_alpha = 0.0
    score_beta = 0.0

    for t in range(1, T):
        sigma2[t] = omega + alpha * eps[t-1]**2 + beta * sigma2[t-1]

        g_omega = 1 + beta * g_omega
        g_alpha = eps[t-1]**2 + beta * g_alpha
        g_beta = sigma2[t-1] + beta * g_beta

        ll_t = -0.5 * (np.log(2 * np.pi) + np.log(sigma2[t]) + eps[t]**2 / sigma2[t])
        loglikelihood += ll_t

        weight = (eps[t]**2 / sigma2[t]**2) - (1 / sigma2[t])

        score_omega += weight * g_omega
        score_alpha += weight * g_alpha
        score_beta += weight * g_beta

    if i % 1000 == 0:
        print(f"Run {i+1}:")
        print(f"  omega={omega:.8f}, alpha={alpha:.4f}, beta={beta:.4f}")
        print(f"  Log-Likelihood = {loglikelihood:.6f}")
        print(f"  Score dL/domega = {score_omega:.6f}")
        print(f"  Score dL/dalpha = {score_alpha:.6f}")
        print(f"  Score dL/dbeta  = {score_beta:.6f}")
        print("-" * 30)

Run 1:
  omega=0.00000092, alpha=0.0978, beta=0.7947
  Log-Likelihood = 11.622662
  Score dL/domega = -8281.604349
  Score dL/dalpha = -2.970568
  Score dL/dbeta  = -1.034776
------------------------------
Run 1001:
  omega=0.00000103, alpha=0.0785, beta=0.8736
  Log-Likelihood = 11.594752
  Score dL/domega = -12788.855342
  Score dL/dalpha = -3.796355
  Score dL/dbeta  = -1.988033
------------------------------
Run 2001:
  omega=0.00000108, alpha=0.1031, beta=0.8267
  Log-Likelihood = 11.592208
  Score dL/domega = -10598.247124
  Score dL/dalpha = -3.404460
  Score dL/dbeta  = -1.600255
------------------------------
Run 3001:
  omega=0.00000091, alpha=0.0930, beta=0.8233
  Log-Likelihood = 11.612735
  Score dL/domega = -10138.618496
  Score dL/dalpha = -3.314051
  Score dL/dbeta  = -1.425606
------------------------------
Run 4001:
  omega=0.00000093, alpha=0.1022, beta=0.8809
  Log-Likelihood = 11.542189
  Score dL/domega = -13142.650177
  Score dL/dalpha = -3.859719
  Score dL/dbet

In [9]:
import numpy as np
from arch import arch_model

returns = np.array([0.01, -0.02, 0.015, -0.005, 0.007])

# Fit GARCH(1,1) model
model = arch_model(returns, vol='Garch', p=1, q=1, dist='normal', mean='Zero')
res = model.fit(disp='off')

print(res.summary())

# Extract parameters and loglikelihood
omega = res.params['omega']
alpha = res.params['alpha[1]']
beta = res.params['beta[1]']
loglikelihood = res.loglikelihood

print(f"\nBuilt-in library results:")
print(f"omega = {omega:.8f}")
print(f"alpha = {alpha:.4f}")
print(f"beta  = {beta:.4f}")
print(f"Log-Likelihood = {loglikelihood:.6f}")


                       Zero Mean - GARCH Model Results                        
Dep. Variable:                      y   R-squared:                       0.000
Mean Model:                 Zero Mean   Adj. R-squared:                  0.200
Vol Model:                      GARCH   Log-Likelihood:                14.8383
Distribution:                  Normal   AIC:                          -23.6765
Method:            Maximum Likelihood   BIC:                          -24.8482
                                        No. Observations:                    5
Date:                Wed, Jul 30 2025   Df Residuals:                        5
Time:                        18:44:01   Df Model:                            0
                               Volatility Model                              
                 coef    std err          t      P>|t|       95.0% Conf. Int.
-----------------------------------------------------------------------------
omega      4.0107e-12  1.353e-06  2.964e-06      1.000 

c:\Users\PanSt\Desktop\UCL\T3\Dissertation\Commodity-Optionality-Pricing-UCL-Dissertation\.venv\Lib\site-packages\arch\univariate\base.py:309: DataScaleWarning: y is poorly scaled, which may affect convergence of the optimizer when
estimating the model parameters. The scale of y is 0.0001578. Parameter
estimation work better when this value is between 1 and 1000. The recommended
rescaling is 100 * y.

This warning can be disabled by either rescaling y before initializing the
model or by setting rescale=False.

  warnings.warn(
